In [ ]:
import torch
from torch import nn
import torchvision
from torchvision import transforms
import numpy as np
import torchvision.models as models

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


In [ ]:
def create_data_loader(batch_size):
  # TODO: create your data loader, use entire CIFAR-10
  # Split the provided CIFAR-10 train set (50,000 images) into your train and val sets
  # Use the first 40,000 images as your train set and the remaining 10,000 images as val set
  # Use all 10,000 images in the provided test set as your test set

  transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
  ])

  trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
  testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)

  train_id = list(range(40000))
  val_id = list(range(40000, 50000))
  test_id = list(range(10000))

  # subset dataset and create dataloader with batch_size=1
  train_sub_set = torch.utils.data.Subset(trainset, train_id)
  val_sub_set = torch.utils.data.Subset(trainset, val_id)
  test_sub_set = torch.utils.data.Subset(testset, test_id)

  train_loader = torch.utils.data.DataLoader(train_sub_set, batch_size=batch_size, shuffle=True)
  val_loader = torch.utils.data.DataLoader(val_sub_set, batch_size=batch_size, shuffle=True)
  test_loader = torch.utils.data.DataLoader(test_sub_set, batch_size=batch_size, shuffle=True)

  return train_loader, val_loader, test_loader

In [ ]:
class ResidualBlock(nn.Module):
  # TODO: implement a residual block with skip connection

  def __init__(self, in_channels, out_channels, stride=1, downSample=None):

    super(ResidualBlock, self).__init__()
    self.conv_layer1 = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, stride=stride, padding=1)
    self.bn1 = nn.BatchNorm2d(out_channels)
    self.relu = nn.ReLU(inplace=True)
    self.conv_layer2 = nn.Conv2d(in_channels=out_channels, out_channels=out_channels, kernel_size=3, stride=stride, padding=1)
    self.bn2 = nn.BatchNorm2d(out_channels)
    self.downSample = downSample

  def forward(self, x):
    identity = x

    out = self.conv_layer1(x)
    out = self.bn1(out)
    out = self.relu(out)

    out = self.conv_layer2(out)
    out = self.bn2(out)

    if self.downSample is not None:
      identity = self.downSample(x)

    out += identity
    out = self.relu(out)

    return out

In [ ]:
class ResNet(nn.Module):
  # TODO: implement a ResNet with residual blocks

  def __init__(self, block, layers, num_classes=10):
    super(ResNet, self).__init__()
    self.in_channels = 64
    self.conv = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3, bias=False)
    self.bn = nn.BatchNorm2d(64)
    self.relu = nn.ReLU(inplace=True)
    self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

    self.layer1 = self.make_layer(block, 64, 64, layers[0])
    self.layer2 = self.make_layer(block, 64, 128, layers[1], stride=1)
    self.layer3 = self.make_layer(block, 128, 256, layers[2], stride=1)
    self.layer4 = self.make_layer(block, 256, 512, layers[3], stride=1)

    self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
    self.fc = nn.Linear(512, num_classes)

  def make_layer(self, block, in_channels, out_channels, blocks, stride=1):
    downSample = None
    if stride != 1 or in_channels != out_channels:
      downSample = nn.Sequential(
          nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
          nn.BatchNorm2d(out_channels)
      )

    layers = []
    # for _ in range(1, blocks):
    layers.append(block(in_channels, out_channels, stride, downSample))
    in_channels = out_channels

    for _ in range(1, blocks):
      layers.append(block(in_channels, out_channels, stride=1))

    return nn.Sequential(*layers)

  def forward(self, x):

      x = self.conv(x)
      x = self.bn(x)
      x = self.relu(x)
      x = self.maxpool(x)

      x = self.layer1(x)
      x = self.layer2(x)
      x = self.layer3(x)
      x = self.layer4(x)

      x = self.avgpool(x)
      x = torch.flatten(x, 1)
      x = self.fc(x)

      return x

In [ ]:
def init_train_var(model):
  # TODO: create your criterion, optimizer (use SGD)

  criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
  optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

  return criterion, optimizer

In [ ]:
def train(train_loader, val_loader, batch_size):
  # TODO: implement your training loop, save your best model at the end

  model = ResNet(ResidualBlock, [2, 2, 2, 2])
  model.to(device)
  model_path = "best_model.pth"

  criterion, optimizer = init_train_var(model)
  best_val_acc = 0.0
  num_epochs = 75

  train_steps = len(train_loader.dataset) // batch_size
  val_steps = len(val_loader.dataset) // batch_size

  for n in range(num_epochs):
    epoch = n
    epoch_train_loss = 0
    train_correct = 0

    epoch_val_loss = 0
    val_correct = 0

    model.train()

    for x,y in train_loader:
      x = x.to(device)
      y = y.to(device)
      optimizer.zero_grad()
      pred = model(x)
      loss = criterion(pred, y)
      loss.backward()
      optimizer.step()

      epoch_train_loss += loss
      train_correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    mean_train_loss = epoch_train_loss / train_steps
    train_acc = train_correct / len(train_loader.dataset)
    print(f"Epoch [{epoch + 1}/{num_epochs}], "
              f"Training Loss: {mean_train_loss:.4f}, Training Accuracy: {train_acc:.2f}")

    with torch.no_grad():
      model.eval()
      for x,y in val_loader:
        x = x.to(device)
        y = y.to(device)
        pred = model(x)
        loss = criterion(pred, y)
        epoch_val_loss += loss
        val_correct += (pred.argmax(1) == y).type(torch.float).sum().item()


    mean_val_loss = epoch_val_loss / val_steps


    val_acc = val_correct / len(val_loader.dataset)

    print(f"Epoch [{epoch + 1}/{num_epochs}], "
              f"Validation Loss: {mean_val_loss:.4f}, Validation Accuracy: {val_acc:.2f}")
    print("\n")


    if val_acc > best_val_acc:
      best_val_acc = val_acc
      torch.save(model.state_dict(), model_path)


In [ ]:
def test(model_path, test_loader):
  # TODO: test function for your trained model , load your best model

  model = ResNet(ResidualBlock, [2, 2, 2, 2])
  model.load_state_dict(torch.load(model_path))
  model.to(device)

  test_correct = 0
  with torch.no_grad():
    model.eval()
    for (x,y) in test_loader:
      x = x.to(device)
      y = y.to(device)
      pred = model(x)
      test_correct += (pred.argmax(1) == y).type(torch.float).sum().item()

    avg_test_acc = test_correct / len(test_loader.dataset)
    print("Test Set Accuracy - " + str(avg_test_acc))

In [ ]:
train_loader, val_loader, test_loader = create_data_loader(batch_size=16)
train(train_loader, val_loader, batch_size=16)
test("best_model.pth", test_loader)

100%|██████████| 170M/170M [00:18<00:00, 9.10MB/s]


Epoch [1/75], Training Loss: 1.7582, Training Accuracy: 0.41
Epoch [1/75], Validation Loss: 1.5383, Validation Accuracy: 0.52


Epoch [2/75], Training Loss: 1.4770, Training Accuracy: 0.56
Epoch [2/75], Validation Loss: 1.4485, Validation Accuracy: 0.57


Epoch [3/75], Training Loss: 1.3513, Training Accuracy: 0.62
Epoch [3/75], Validation Loss: 1.3006, Validation Accuracy: 0.64


Epoch [4/75], Training Loss: 1.2516, Training Accuracy: 0.67
Epoch [4/75], Validation Loss: 1.2617, Validation Accuracy: 0.67


Epoch [5/75], Training Loss: 1.1718, Training Accuracy: 0.71
Epoch [5/75], Validation Loss: 1.2338, Validation Accuracy: 0.69


Epoch [6/75], Training Loss: 1.0992, Training Accuracy: 0.75
Epoch [6/75], Validation Loss: 1.1992, Validation Accuracy: 0.70


Epoch [7/75], Training Loss: 1.0356, Training Accuracy: 0.78
Epoch [7/75], Validation Loss: 1.2256, Validation Accuracy: 0.70


Epoch [8/75], Training Loss: 0.9716, Training Accuracy: 0.81
Epoch [8/75], Validation Loss: 1.1974, Vali